In [46]:
import pandas as pd
import numpy as np
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"
df = pd.read_csv(url)




In [47]:
def size_transform(size):
    #Se tienen que limpiar más los datos porque el dataset contiene entradas como 1,000+
    size = size.replace('+', '').replace(',', '')

    if size == 'Varies with device':
        return np.nan
    elif size[-1] == 'M':
        return float(size[:-1]) * 1024 * 1024
    elif size[-1] == 'k':
        return float(size[:-1]) * 1024
    else:
        return float(size)

In [48]:
df['Size'] = df['Size'].apply(size_transform)

print(df['Size'].head())

0    19922944.0
1    14680064.0
2     9122611.2
3    26214400.0
4     2936012.8
Name: Size, dtype: float64


In [49]:
df['Size'].mean()

np.float64(22558867.99930024)

 ¿Cuál es el peso promedio en Megabytes de las apps en la Play Store?
 El resultado aproximado es 22,558,868 de bytes o 22.51MB (dividiendo por 1024 para convertirlo)

In [50]:
print(df['Last Updated'].head())


0     January 7, 2018
1    January 15, 2018
2      August 1, 2018
3        June 8, 2018
4       June 20, 2018
Name: Last Updated, dtype: object


In [51]:
#Se debe convertir la columna porque contiene valores como 1.0.19 que no concuerda con el formato de la función
def convertir_fecha(valor):
    formatos = [
        '%B %d, %Y',  # January 7, 2018
        '%d/%m/%Y',   # 07/01/2018
        '%Y-%m-%d',   # 2018-01-07
        '%d.%m.%y'    # 7.1.18
    ]

    for formato in formatos:
        try:
            return pd.to_datetime(valor, format=formato)
        except (ValueError, TypeError):
            pass

    return pd.NaT

df['Last Updated'] = df['Last Updated'].apply(convertir_fecha)

In [52]:
print(df['Last Updated'].head())

0   2018-01-07
1   2018-01-15
2   2018-08-01
3   2018-06-08
4   2018-06-20
Name: Last Updated, dtype: datetime64[ns]


In [53]:
df['Year_Updated'] = df['Last Updated'].dt.year
print(df['Year_Updated'].head())

0    2018.0
1    2018.0
2    2018.0
3    2018.0
4    2018.0
Name: Year_Updated, dtype: float64


In [54]:
df.value_counts(subset=['Year_Updated'])

Year_Updated
2018.0          7349
2017.0          1867
2016.0           804
2015.0           459
2014.0           209
2013.0           110
2012.0            26
2011.0            15
2010.0             1
Name: count, dtype: int64

 ¿en qué año se actualizó la mayor cantidad de aplicaciones en nuestro dataset?
 En 2018 con 7349 actualizaciones

In [55]:
df['Size'].isna().sum()

np.int64(1695)

¿es matemáticamente más sano borrar esas filas con un .dropna() porque el peso de la app es crítico, o prefieren rellenar ese hueco imputando la mediana global del peso de las apps?

Al contar los registros que terminaron con Nan, salieron 1695, los cuales son una gran cantidad de registros, por lo que mejor conviene rellenarlo con la mediana.

In [56]:
mediana = df['Size'].median()
df['Size'].fillna(mediana)

0        19922944.0
1        14680064.0
2         9122611.2
3        26214400.0
4         2936012.8
            ...    
10836    55574528.0
10837     3774873.6
10838     9961472.0
10839    13631488.0
10840    19922944.0
Name: Size, Length: 10841, dtype: float64

Decidí rellenar los valores Nan con la mediana de Size para evitar eliminar 1695 registros de información potencialmente útil.
Se decidió este método porque la mediana es resistente a el sesgo por outliers, además de que ayuda a representar mejor el 
valor típico de las aplicaciones. 